# 09. What moves the 49 industries together: principal components

**Terms used in this notebook.**

| Term | Meaning |
|---|---|
| Risk-free rate | the return on a one-month US Treasury bill, the closest thing to a return with no risk |
| Excess return | a return minus the risk-free rate over the same month |
| Market capitalisation | the number of a firm's shares times their price, the firm's size in money |
| Value-weighted return | the return of a group of firms in which each firm counts in proportion to its market capitalisation at the start of the month |
| Cap-weighted | weighted by market capitalisation |
| Benchmark | in the extension, the index the portfolio tracks, the cap-weighted combination of the 49 industries built in notebook 08 |
| Fat tails | extreme months more frequent and more extreme than a normal distribution allows |
| Volatility clustering | volatility that persists from month to month, so that a turbulent month is usually followed by another |
| Variance | the average squared distance of a series from its own average; its square root is the standard deviation, the usual measure of how much a return moves |
| Covariance | a number that says how two series move together, positive when they tend to be above their averages in the same months and negative when one is above while the other is below; the covariance of a series with itself is its variance |
| Covariance matrix | the table of all variances and covariances of a set of assets, 49 rows by 49 columns here, 1,225 distinct numbers |
| Principal component | a combination of the assets, with one weight per asset, chosen so that it explains as much of the assets' total variance as a single combination can; the second component does the same among combinations uncorrelated with the first, and so on; in mathematics the components are called the eigenvectors of the covariance matrix and their variances are called its eigenvalues |
| Loading | an asset's weight in a component |
| Score | a component's monthly return, the assets' excess returns combined with the loadings as weights |
| Variance share | the component's variance divided by the total variance of all assets; the shares of all 49 components add up to one |
| Seed | the number that fixes the random numbers a simulation uses; the same seed gives the same random numbers |
| Draw | one panel of made-up returns produced from one seed |
| Parallel analysis | the comparison of each component's variance with the variance the same component would have if the assets had nothing in common and moved independently; a component above that bar carries shared movement that chance cannot imitate |
| Estimation error | the difference between a number estimated from a window of returns and its true value |
| Standard error | the typical size of the estimation error in an estimate; for a correlation measured on T months it is about (1 minus the squared correlation) divided by the square root of T |
| Estimation window | the M months of past returns a rule sees when it forms its weights, 120 in the extension |
| Factor | a return series that moves many assets at once, for example the return of the whole market |
| Beta | how much an asset moves with a factor; an industry with a beta of 1.5 to the benchmark moves 1.5 times as much as the benchmark on average |
| Idiosyncratic | an asset's own movement, unrelated to the factors |
| Covariance estimator | a method for estimating the covariance matrix from a window of data; the extension compares four in notebook 10 |
| Vintage | the version of Ken French's files on the download date, named by the release of the CRSP database they were built from |
| Provenance | the record of what was downloaded, when, with its checksum and vintage |

**The problem, stated the way practitioners state it.** The key ingredient of every risk-based portfolio is the covariance matrix of the assets' returns. The natural estimate is the sample covariance, and it fails when the assets are many relative to the months of data: for 49 industries and 120 months it has fewer than five observations per number, and the rule that inverts it (notebooks 02 and 03) trades on its errors. The literature answers with a long list of alternative estimators, and the three ideas behind them are shrinkage (pulling the estimate part of the way towards a simpler target), time dynamics (weighting recent data more) and factor structure (Dom, Howard, Jansen and Lohre, 2024, whose introduction this paragraph follows). Factor structure is the idea this notebook prepares: describe the 1,225 numbers by five or fewer common sources of movement plus each industry's own variance, so that fewer numbers are estimated from the same data. Whether that is a good idea for the 49 industries depends on a fact about the data, whether five sources account for most of their shared movement and whether the first of them is the market, which is what this notebook measures before notebook 10 builds the estimator on it.

**What this notebook does.** The paper estimates the covariance matrix from a 120-month window of monthly returns. For 49 industries that is 49 variances and 1,176 covariances, 1,225 numbers, from 49 times 120 equals 5,880 monthly returns: fewer than five observations per number. Notebooks 02 and 03 showed what estimation error does to a rule that inverts such a matrix. Practitioners meet this problem in two ways, and the extension uses both. The first is more observations: estimating from daily returns, where three years hold about 756 trading days and 49 times 756 equals 37,044 observations, 30 per number; notebook 10 builds every estimator at both frequencies, the paper's 120 monthly returns and three years of daily returns, and `constants.py` records the design (`EXT_COV_FREQUENCIES`). The second is fewer numbers: describe the matrix by the few combinations of industries that account for most of the shared movement, keep those, and treat what is left as each industry's own noise. Those combinations are the principal components, and they are this notebook's subject. The components here are taken from monthly returns, the paper's data, so that what they show about the industries' structure is the structure the paper's estimator faced. Before that estimator is built, this notebook asks three things of the 49 industries over 1969-07 to the vintage's last month: how much of the total variance the first few components explain; whether the first component is the market, measured as the correlation of its score with the benchmark's excess return; and whether the components found in one period are still the components of another, because an estimator that keeps five components assumes that the five found in the window still describe the next month.

**Why it comes ninth.** The benchmark exists since notebook 08, and the first component is checked against it. The question of how many numbers a covariance matrix really needs is the bridge from the paper (where the sample covariance was one of the things that made mean-variance fail) to the extension (where four ways of estimating it are compared on a portfolio that is told what to hold).

**What I expect to see, written before the run (`constants.py`, 1 October 2026).** First, count first: 686 months by 49 industries with no gap, and the benchmark rebuilt from the same module over the same months. Second, the first component is the market: its score has a correlation of at least 0.95 with the benchmark's excess return over the full sample and in each of three windows fixed in advance (to 2019-12, 2020-01 to 2022-12, from 2023-01). Below the level in any window the notebook reports which, and notebook 10's PCA estimator is built with the market factor imposed as the first component. Third, reported with no pass or fail: the variance share of each of the first ten components, the share of the first five together (the number notebook 10 keeps), and the number of components above the parallel-analysis bar. Fourth, reported: the stability of the first five components' loadings between the three windows, and how much of each window's variance the full-sample components explain compared with the window's own. Fifth, reported: what the estimator will see, the first component's variance share and its correlation with the benchmark in every 120-month window. Running time: about two minutes, most of it the parallel analysis.

## Modules

`constants`, `french_loader` and `benchmark` are those of notebook 08. `pca` is new: the components with a sign rule, their scores and variance shares, a similarity measure between two sets of loadings, the parallel analysis, and the rolling view. Each function has a test on a made-up one-factor market in `tests/test_pca.py`.

In [ ]:
import os
os.makedirs("src/bp", exist_ok=True)
open("src/bp/__init__.py", "w").close()

In [ ]:
%%writefile src/bp/constants.py
"""
Pre-committed numbers for the study.

Every band, gate, sample boundary and design parameter is defined here, one
per line, with its source beside it. A notebook may read these names; it may
not introduce numbers of its own. The replication bands (section 3) and the
design of the extension (section 5) were fixed on 10 September 2026, before any
of the rules' code was written. The expectations each notebook tests are
stated in that notebook's introduction and were written before its code; where
a level was changed after a run, the comment beside it says so, with the date
and the reason, so that a reader can judge whether the change was a
specification settled by the paper's own text or a number tuned to the results.

DGU throughout means DeMiguel, Garlappi and Uppal (2009), "Optimal Versus
Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?",
Review of Financial Studies 22(5), 1915-1953. Page numbers are the journal's.
"""

# ---------------------------------------------------------------------------
# 1. Data: Ken French's library, downloaded at run time, never committed
# ---------------------------------------------------------------------------

FRENCH_BASE_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"

# Every file the study reads. The vintage is whatever the library serves on the
# download date; the loader records the date, the SHA-256 of the zip and the
# CRSP vintage line printed at the top of each CSV.
FRENCH_FILES = {
    "factors3":   "F-F_Research_Data_Factors_CSV.zip",        # Mkt-RF, SMB, HML, RF, monthly from 1926-07
    "momentum":   "F-F_Momentum_Factor_CSV.zip",              # Mom, monthly from 1927-01
    "factors5":   "F-F_Research_Data_5_Factors_2x3_CSV.zip",  # Mkt-RF, SMB, HML, RMW, CMA, RF from 1963-07
    "ind10":      "10_Industry_Portfolios_CSV.zip",           # DGU "Industry" dataset
    "ind49":      "49_Industry_Portfolios_CSV.zip",           # the extension universe; carries firm counts and average size
    "size_bm25":  "25_Portfolios_5x5_CSV.zip",                # DGU "FF-1-factor" and "FF-4-factor" datasets
    # Daily files, for the extension's covariance estimators (section 5).
    "ind49_daily":    "49_Industry_Portfolios_daily_CSV.zip",        # value- and equal-weighted daily returns from 1926-07-01
    "factors3_daily": "F-F_Research_Data_Factors_daily_CSV.zip",     # Mkt-RF, SMB, HML, RF daily from 1926-07-01
    "factors5_daily": "F-F_Research_Data_5_Factors_2x3_daily_CSV.zip", # Mkt-RF, SMB, HML, RMW, CMA, RF daily from 1963-07-01
    "momentum_daily": "F-F_Momentum_Factor_daily_CSV.zip",           # Mom daily from 1926-11-03
}

FRENCH_MISSING_CODES = (-99.99, -999.0)   # French's own convention, stated at the top of every portfolio file
FRENCH_RETURNS_ARE_PERCENT = True         # the loader divides by 100 once, at load time, and nowhere else

# ---------------------------------------------------------------------------
# 2. The DGU replication
# ---------------------------------------------------------------------------

DGU_SAMPLE_START = "1963-07"   # DGU Table 2, p. 1918, for every French-sourced dataset
DGU_SAMPLE_END = "2004-11"     # DGU Table 2, p. 1918
DGU_ESTIMATION_WINDOW = 120    # months; DGU section 2, p. 1927. M = 60 is in DGU's appendix and is out of scope.
DGU_RISK_AVERSION = 1.0        # gamma in the CEQ, DGU p. 1929: "the results we report are for the case of gamma = 1"
DGU_TCOST = 0.0050             # proportional cost per unit of one-way turnover; DGU p. 1929: 50 basis points, after Balduzzi and Lynch (1999)
TCOST_SENSITIVITY = 0.0100     # the one cost sensitivity reported: double the base cost
DGU_GMINC_LOWER_BOUND_FRACTION = 0.5   # g-min-c imposes w >= a * 1 with a = 1/(2N); DGU p. 1926
DGU_BS_COV_DOF = "M - N - 2"   # the Bayes-Stein covariance divides by M - N - 2; DGU equation (5), p. 1923

# Settled 22 September 2026, a specification the paper leaves ambiguous. DGU's
# Lagrangian (8), p. 1925, shows the short-sale constraint alone; footnote 22,
# p. 1934, says the constrained rules produce "corner solutions with all wealth
# invested in a single asset", which only happens when the budget constraint
# 1'w = 1 sits inside the optimisation with gamma = 1. With the budget inside,
# mv-c on MKT/SMB/HML gives a Sharpe ratio of 0.1093 against the published
# 0.1084 and a corner solution in 257 of 377 months; without it, 0.2382 and no
# corner. The reading that reproduces the paper's tables is used; the literal
# reading is kept as rules.mv_c_cone for comparison (notebook 03).
DGU_CONSTRAINED_BUDGET_INSIDE = True

# The rules replicated: those whose only inputs are the window's sample moments
# (and, for Bayes-Stein, Jorion's prior built from them), plus the value-weighted
# market, whose Sharpe ratio is the same 0.1138 in every French-sourced column
# of Table 3 and so checks that the market series and the sample match the paper's.
DGU_RULES = ("ew", "mv", "bs", "min", "vw", "mv-c", "bs-c", "min-c", "g-min-c")
DGU_IN_SAMPLE_ROW = "mv (in sample)"   # DGU Table 3 row 2: mean-variance with M = T, the upper bound estimation error costs

# Excluded, each with the criterion. They are not built.
DGU_RULES_EXCLUDED = {
    "Bayesian diffuse-prior": "not reported by DGU (Table 1 note, p. 1917)",
    "dm": "Bayesian Data-and-Model (Pastor 2000); needs an asset-pricing prior, outside the moment-based set",
    "mp": "MacKinlay and Pastor (2000) missing-factor model; needs a maximum-likelihood factor structure, outside the moment-based set",
    "mv-min": "Kan and Zhou (2007) three-fund mixture of two rules already built; not part of the pre-specified set",
    "ew-min": "mixture of 1/N and minimum variance; not part of the pre-specified set",
    "multi-prior": "not reported by DGU (Table 1 note, p. 1917)",
}

# The datasets replicated: the four DGU built from French's site (DGU Table 2, p. 1918; Appendix A, pp. 1948-1949).
# Each is monthly excess returns over the T-bill rate from French's factor file.
DGU_DATASETS = {
    "Industry":     {"n_risky": 10, "factors": ("Mkt-RF",), "N": 11, "source": "ind10 + factors3"},
    "MKT/SMB/HML":  {"n_risky": 0,  "factors": ("Mkt-RF", "SMB", "HML"), "N": 3, "source": "factors3"},
    "FF-1-factor":  {"n_risky": 20, "factors": ("Mkt-RF",), "N": 21, "source": "size_bm25 + factors3"},
    "FF-4-factor":  {"n_risky": 20, "factors": ("Mkt-RF", "SMB", "HML", "Mom"), "N": 24, "source": "size_bm25 + factors3 + momentum"},
}
DGU_DATASETS_EXCLUDED = {
    "S&P Sectors": "Roberto Wessels' GICS sector data, not on French's site (DGU Appendix A.1)",
    "International": "MSCI country indices, not on French's site (DGU Appendix A.3)",
    "FF-3-factor": "DGU do not report it: 'very similar to FF-1-factor' (Table 3 note, p. 1931)",
}
DGU_EXCLUDE_LARGEST_SIZE_QUINTILE = True   # DGU footnote 24, p. 1949: the five largest-size portfolios of the 25 are dropped, leaving 20

# ---------------------------------------------------------------------------
# 3. Bands for the replication, written before the code ran
# ---------------------------------------------------------------------------

BAND_SHARPE_ABS = 0.03          # out-of-sample Sharpe ratio within 0.03 of DGU Table 3, per rule and dataset
CHECK_TOP3_RANKING = True        # the ranking of the top three rules within each dataset is preserved
BAND_TURNOVER_REL = 0.25        # turnover within 25% relative of DGU Table 5

# DGU Table 5 reports the unconstrained mean-variance rule's turnover at 606,594
# times that of 1/N on the Industry dataset and 10,466 times on FF-1-factor.
# Those figures are dominated by the months in which the sample covariance is
# near-singular and move by orders of magnitude when French revises a single
# return, so the band applies only where DGU's published relative turnover is
# below the level here; above it, the replicated figure is reported beside the
# published one with no pass or fail. Fixed 10 September 2026.
BAND_TURNOVER_APPLIES_BELOW_RELATIVE = 100.0

# Changed 13 September 2026, after notebook 02 had run, on its evidence: the
# out-of-sample Sharpe band applies only to rules whose published relative
# turnover (DGU Table 5, panel A) is below the same level, and the in-sample
# mean-variance row carries no verdict. Evidence: on the Industry, FF-1 and FF-4
# datasets the unconstrained mean-variance rule moved by 0.12 to 0.25 in Sharpe
# ratio under ten basis points of noise, with the published figure inside the
# range, while the 1/N rows matched to the third decimal. Before the change the
# band applied to every out-of-sample row. The level is the paper's own number,
# already used by the turnover band.
BAND_SHARPE_APPLIES_BELOW_RELATIVE = BAND_TURNOVER_APPLIES_BELOW_RELATIVE
BAND_SHARPE_DECISION_DATE = "2026-09-13"

# Sharpe ratios are compared on the same sample, 1963-07 to 2004-11, 120-month
# window, so the out-of-sample period is 1973-07 to 2004-11.
DGU_EXPECTED_T = 497           # months from 1963-07 to 2004-11 inclusive; a count, checked in notebook 01
DGU_EXPECTED_OOS = 377         # DGU_EXPECTED_T - DGU_ESTIMATION_WINDOW

# The replication extended to the latest month (notebook 07), fixed 1 October
# 2026 before the notebook's code: the nine rules on every month the vintage
# carries, reported for the paper's out-of-sample period, the months since the
# paper and the whole period. The expectations are in the notebook's introduction.
EXTENDED_NEW_PERIOD_START = "2004-12"   # the month after DGU's sample ends

# ---------------------------------------------------------------------------
# 4. DGU's analytical result and simulation (notebook 04)
# ---------------------------------------------------------------------------

# The analytical critical window (DGU Proposition 1, pp. 1937-1938; Figure 1, p. 1940).
# Panel E is the one calibrated to US stock-market data and the one the abstract quotes.
ANALYTIC_SHARPE_TANGENCY = 0.15    # S*, DGU Figure 1 panel E
ANALYTIC_SHARPE_1N = 0.12          # S_ew, DGU Figure 1 panel E
EXPECT_CRITICAL_M_LOWER = {25: 3000, 50: 6000}   # DGU p. 1941: "more than 3000 months" and "more than 6000 months"; the check is order of magnitude and direction in N

# The simulated dataset (DGU section 5.1, pp. 1941-1942).
SIM_RF_MEAN_ANNUAL = 0.02
SIM_RF_SD_ANNUAL = 0.02
SIM_FACTOR_MEAN_ANNUAL = 0.08
SIM_FACTOR_SD_ANNUAL = 0.16
SIM_ALPHA = 0.0
SIM_BETA_RANGE = (0.5, 1.5)          # loadings evenly spread
SIM_IDIO_VOL_RANGE = (0.10, 0.30)    # annual idiosyncratic volatility, uniform, cross-sectional mean 0.20
SIM_N = (10, 25, 50)
SIM_M = (120, 360, 6000)
SIM_T = 24000                        # months in one simulated history
SIM_SEED = 20260910                  # fixed so the run is reproducible; DGU's seed is unknown
# In one 24,000-month history the standard error of a monthly Sharpe ratio is
# about 1/sqrt(24000) = 0.0065, so the comparison with DGU Table 6 uses the same
# band as the empirical tables, plus the sign of (mv minus 1/N) in every cell.
BAND_SIM_SHARPE_ABS = BAND_SHARPE_ABS
# Table 6 reports 1/N and "mv (true)" as full-sample Sharpe ratios and the
# estimated rules out of sample; the comparison follows that convention. As a
# second check every rule is rerun on alternative draws of the market (seeds 1
# to SIM_SPREAD_DRAWS) and the published value is checked against their range.
SIM_SPREAD_DRAWS = 8

# Fat tails (notebook 05), an addition to the paper, fixed 30 September 2026
# before the notebook's code. The market above is rerun with every shock drawn
# from a Student-t distribution, scaled so that the mean and covariance of
# returns are exactly those of the normal market; one scale draw per month is
# shared by all shocks (a multivariate t), so an extreme month hits all assets
# at once. The normal shocks are the same draws as notebook 04's.
SIM_T_DOF = (10, 5)                  # degrees of freedom. 10: excess kurtosis 1, about a broad index's monthly returns. 5: excess kurtosis 6, about an industry portfolio's. At 4 or below the fourth moment is infinite and the standard error of a Sharpe ratio is undefined.
SIM_T_COMMON_SCALE = True            # one scale per month shared by all shocks (multivariate t); False draws one scale per shock series, which changes the shape of the month's co-movement as well as its size
# The crossing table: sample-based mean-variance against 1/N over a grid of
# windows, to locate the window at which the gap closes under each distribution.
SIM_CROSSING_WINDOWS = (120, 240, 360, 600, 1000, 2000, 3000, 4000, 6000, 9000, 12000)
FAT_TAIL_MOMENT_CHECK_ABS = 0.013    # notebook 05, expectation 1: two standard errors of a 24,000-month Sharpe ratio
FAT_TAIL_DIRECTION_SLACK = 0.01      # notebook 05, expectation 2: a cell counts as in the expected direction if Sharpe(t) <= Sharpe(normal) + this
# One test added after notebook 05's first run (6 October 2026), fixed here before
# its code and labelled in the notebook: the fat-tailed market rerun with one
# scale per shock series (common_scale False), so that an extreme month changes
# the shape of the estimated covariance as well as its level. Expectation P1a:
# the mean change in Sharpe ratio across the seven estimated rules and the nine
# cells of the Table 6 grid is more negative than under the common scale at both
# degrees of freedom. P1b: the rules whose weights depend on the shape of the
# covariance and use no means lose more on average than the other estimated rules.
FAT_TAIL_SHAPE_RULES = ("min", "min-c", "g-min-c")   # P1b: the minimum-variance rules

# Volatility that changes through time (notebook 06), the second addition to
# the paper, fixed 1 October 2026 before the notebook's code. The market of
# notebook 04 is rerun with shocks whose volatility follows a GARCH(1,1) process
# (Bollerslev 1986); the factor shock and every asset's idiosyncratic shock get
# their own process with the same parameters, so both the level and the shape
# of the covariance matrix move through time, while the unconditional mean and
# covariance are exactly those of the normal market (omega = 1 - alpha - beta).
# The expectations of notebooks 05 and 06 are in their introductions and use
# the two levels above.
SIM_GARCH_ALPHA = 0.10               # weight of last month's squared shock; a textbook monthly value
SIM_GARCH_BETA = 0.85                # weight of last month's variance; alpha + beta = 0.95 is the persistence, a half-life of a volatility shock of about 13 months
SIM_GARCH_OWN_PROCESS_PER_ASSET = True   # the factor and each idiosyncratic shock follow their own process; False would scale all shocks by one common volatility, which leaves the shape of the covariance unchanged

# ---------------------------------------------------------------------------
# 5. The extension: constrained index tracking on the 49 industries
# ---------------------------------------------------------------------------

EXT_UNIVERSE = "ind49"
EXT_ESTIMATION_WINDOW = 120        # the same window as the replication, so one convention runs through the repository
EXT_REBALANCE = "monthly"
# The extension sample starts at the first month in which all 49 industries carry
# a return (French codes the early gaps as -99.99): Hlth is the last industry to
# fill and does so in 1969-07. Notebook 01 asserts it on every run.
EXT_SAMPLE_START = "1969-07"
EXT_SAMPLE_END = None              # the latest month the vintage carries

# The benchmark (the index the portfolio tracks): the cap-weighted combination
# of the 49 industries, weights from firm count times average firm size, must
# track the market factor plus the risk-free rate within this tracking error a
# year over the full sample (notebook 08). If it does not, the derived weights
# are dropped and the benchmark is the market series.
CAPW_TE_MAX_ANNUAL = 0.0050

# Notebook 08, the cap-weight check, fixed 1 October 2026 before its code. The
# weight of industry i in month t is its firm count times its average firm size
# in French's file for month t, divided by the sum over the 49 industries;
# French measures both at the start of month t, so the weight uses nothing
# unknown when the month begins. The previous-month reading is run beside it as
# a check of this timing, and the notebook stops if it tracks more closely.
CAPW_WEIGHT_TIMING = "same_month"
CAPW_WEIGHT_TIMING_ALTERNATIVE = "previous_month"   # reported beside the chosen timing, never used as the benchmark
CAPW_MARKET_SERIES = ("Mkt-RF", "RF")                # the market series is Mkt-RF plus RF, so that both sides are plain returns
CAPW_REPORT_MONTHS = ("first", "2004-11", "last")

# The direct test of where the benchmark's gap to the market comes from
# (notebook 08, second part), fixed 1 October 2026 before its code. French forms
# the industry portfolios once a year at the end of June from the firms that
# have an SIC code then, while the market return takes in a firm from its first
# full month of trading, so between one July and the next the market holds
# firms that no industry portfolio holds. The test needs the firm list of each
# portfolio each month, which French does not publish, so it runs on the CRSP
# monthly stock file (WRDS, under Queen Mary University of London's
# subscription), the one use of data outside French's library in the study.
# The CRSP data stay on the machine that holds them: no notebook reads them,
# the repository carries none, and only the aggregates in
# DIRECT_TEST_RESULTS_FILE are published, with CRSP cited by name. The market
# universe is CRSP share codes 10 and 11 on NYSE, AMEX and NASDAQ, weighted by
# start-of-month market equity; the June-formed universe is the firms of that
# market that were in it at the end of June of the formation year with a
# positive SIC code. The script is checks/crsp_direct_test.py; its pure
# functions are in bp.direct_test.
DIRECT_TEST_SHARE_CODES = (10, 11)
DIRECT_TEST_EXCHANGE_CODES = (1, 2, 3)
DIRECT_TEST_FORMATION_MONTH = 6          # the end of June
DIRECT_TEST_MARKET_TE_MAX = 0.0020       # expectation 1: the CRSP market universe tracks French's market return within 20 basis points a year
DIRECT_TEST_MIN_CORRELATION = 0.80       # expectation 2: correlation of the predicted gap with the observed one
DIRECT_TEST_SLOPE_RANGE = (0.8, 1.2)     # expectation 2: slope of the observed gap on the predicted one
DIRECT_TEST_RESIDUAL_TE_MAX = 0.0025     # expectation 4: tracking error of what the explanation leaves, 25 basis points a year
DIRECT_TEST_RESULTS_FILE = "checks/results_crsp_direct_test.json"   # aggregates only; committed

# Constraints. Fixed 10 September 2026; the sources for the levels are in
# notebook 11, "Where the numbers come from".
ACTIVE_WEIGHT_BOUND = 0.02         # each untilted industry within 2 percentage points of its benchmark weight; an absolute cap does not fit 49 industries whose benchmark weights run from under 0.5% to over 10%
TURNOVER_CAP_MONTHLY_ONE_WAY = 0.02   # one-way turnover per rebalance, against drifted weights; 2% a month is about 24% a year, the enhanced-indexing range
TILT_INDUSTRIES = ("Coal", "Oil", "Util")   # French's column names, stripped of padding
TILT_MAX_SHARE_OF_BENCHMARK = 0.5  # each tilted industry held at no more than half its benchmark weight, a 50% reduction relative to the benchmark; a tilted industry's bounds are [0, 0.5 * benchmark], and ACTIVE_WEIGHT_BOUND does not apply to it, because the two would conflict for any industry above 4% of the benchmark
LONG_ONLY = True

# Constraint sets, cumulative, reported in this order. C3 is the mandate: the
# tilt with beta neutrality and the exclusions. Beta neutrality and the
# exclusions were added on 4 October 2026, after notebook 11's checks D1 to D4
# and before notebook 12's code, because the tilt's hedge left a beta bet
# standing when the covariance was noisy and bought tobacco and weapons. The
# tilt alone is CONSTRAINT_SET_TILT_ONLY, the C3 of the first design, reported
# beside the mandate in notebooks 11 and 12 so that the cost of the two
# additions is visible.
CONSTRAINT_SETS = {
    "C0": ("long_only",),
    "C1": ("long_only", "active_weight_bound"),
    "C2": ("long_only", "active_weight_bound", "turnover_cap"),
    "C3": ("long_only", "active_weight_bound", "turnover_cap", "tilt", "beta_neutral", "exclusion"),
}
CONSTRAINT_SET_TILT_ONLY = ("long_only", "active_weight_bound", "turnover_cap", "tilt")   # the C3 of the design of 10 September 2026
MANDATE_EXCLUSIONS = ("Smoke", "Guns")   # tobacco and weapons, the standard exclusions of Dutch institutional mandates, in French's industry names; weight zero

# The data the covariance estimators see, fixed 1 October 2026. The 120-month
# window is the paper's convention and gives 49 x 120 = 5,880 returns for the
# 1,225 entries of a 49-industry covariance matrix, fewer than five per entry.
# Practitioners estimate from daily returns (three years hold about 756 trading
# days, 30 observations per entry), impose structure, and weight recent days
# more; the extension does all three. Each estimator is built at two data
# frequencies: "monthly_120", the last EXT_ESTIMATION_WINDOW months of monthly
# excess returns, and "daily_3y", the last EXT_COV_DAILY_WINDOW_YEARS years of
# daily excess returns scaled to a monthly covariance by DAILY_TO_MONTHLY_SCALE
# (notebook 10 tests that scaling). A trading day on which any industry has no
# return (one in the sample, Softw on 1971-03-11) is left out of the window.
EXT_COV_FREQUENCIES = ("monthly_120", "daily_3y")
EXT_COV_DAILY_WINDOW_YEARS = 3
EXT_COV_DAILY_WINDOW_YEARS_SENSITIVITY = (1, 5)   # reported for the sample estimator only, no verdict
DAILY_TO_MONTHLY_SCALE = 21                        # 252 trading days a year divided by 12
EXT_COV_EWMA_HALFLIFE_DAYS = 252                   # the recency-weighted sensitivity: a half-life of one year of trading days, the RiskMetrics setting of Dom, Howard, Jansen and Lohre (2024, section 3.3.2); 126 until 3 October 2026, changed on reading that paper, before notebook 10's code

# Covariance estimators. The shrinkage target follows Dom, Howard, Jansen and
# Lohre (2024), the paper the design named as its reference, checked on
# 3 October 2026 before notebook 10's code: their linear shrinkage pulls the
# sample covariance towards a scaled identity matrix (Ledoit and Wolf 2004b),
# and their footnote 11 reports that the constant-correlation target gave
# inferior results. The same paper estimates from three years of daily returns
# and rebalances monthly, and finds that under long-only constraints the
# differences between estimators shrink, which is notebook 10's expectation 3.
COV_ESTIMATORS = ("sample", "ledoit_wolf", "factor", "pca")
LW_TARGET = "scaled_identity"          # Ledoit and Wolf (2004b), "A well-conditioned estimator for large-dimensional covariance matrices"; was "constant_correlation" until 3 October 2026
LW_TARGET_SENSITIVITY = "constant_correlation"   # Ledoit and Wolf (2004a), "Honey, I Shrunk the Sample Covariance Matrix"; reported, no verdict
FACTOR_MODEL_FACTORS = ("Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom")   # the Fama-French five factors plus momentum; the estimator is B F B' plus a diagonal of residual variances, B estimated in the same estimation window
PCA_N_COMPONENTS = 5                   # five components plus a diagonal idiosyncratic term
PCA_PC1_MARKET_CORR_MIN = 0.95         # notebook 09, expectation 2: the first component's correlation with the cap-weighted market return
PCA_USE_COVARIANCE = True              # covariance, not correlation, of monthly excess returns
PCA_SIGN_RULE = "positive market loading"
PCA_WINDOWS = {                        # three pre-specified windows; loading stability is the finding and carries no pass or fail
    "to_2019":   (None, "2019-12"),
    "2020_2022": ("2020-01", "2022-12"),
    "from_2023": ("2023-01", None),
}

# Notebook 09, the principal components of the 49 industries, fixed 1 October
# 2026 before its code; the expectations are in its introduction.
PCA_REPORT_COMPONENTS = 10
PCA_PARALLEL_DRAWS = 200
PCA_PARALLEL_PERCENTILE = 95
PCA_PARALLEL_SEED = 20260910
# Set after notebook 09 ran, as its design said: the first component's
# correlation with the benchmark fell below PCA_PC1_MARKET_CORR_MIN in the
# window from 2023 (0.837), so notebook 10's PCA estimator takes the
# benchmark's excess return as its first factor. Parallel analysis found two
# components above noise; PCA_N_COMPONENTS stays at five and the two-component
# version is reported beside it.
PCA_FIRST_COMPONENT = "market"        # notebook 10: the benchmark's excess return is the first factor; "data" would take it from the eigenvectors
PCA_N_COMPONENTS_SENSITIVITY = 2      # the parallel-analysis count on the full sample, reported beside PCA_N_COMPONENTS

# Notebook 10, the four covariance estimators, fixed 3 October 2026 before its
# code. The estimators, data frequencies, tests and expectations are stated in
# the notebook's introduction and in covariance.py.
COV_EVAL_START = "1979-07"            # 1969-07 plus 120 months; the three-year daily window starts 1972-07 and is the shorter requirement
COV_TEST_ACTIVE_DRAWS = 200           # random active-weight vectors for the calibration test
COV_TEST_SEED = 20260910
COV_LONG_ONLY_RANGE_MAX = 0.01        # expectation 3: one percentage point of annualised volatility across the four estimators on daily data
COV_SCALE_RATIO_RANGE = (0.8, 1.25)   # expectation 4: the ratio of the benchmark's realised monthly variance to 21 times its daily variance
# Three tests added after notebook 10's first run (4 October 2026), each fixed
# here before its code and labelled in the notebook: P1, a scale correction by
# the variance ratio of Lo and MacKinlay (1988), one number from the benchmark's
# daily returns that multiplies the daily covariance in place of
# DAILY_TO_MONTHLY_SCALE alone; P2, bias statistics by decade; P3, trading costs
# of the minimum-variance portfolios at EXT_COST_LEVELS.
COV_SCALE_VR_HORIZON_DAYS = DAILY_TO_MONTHLY_SCALE   # the horizon of the variance ratio: one month of trading days
COV_SCALE_VR_WINDOW_YEARS_SENSITIVITY = 10           # P1 sensitivity: the variance ratio from ten years of daily benchmark returns
COV_POST_RUN_DECADES_MIN_CLOSER = 4                  # P1a: of five decades, the corrected forecast is closer to calibrated in at least this many
COV_POST_RUN_COST_MIN_VARIANTS = 6                   # P3a and P3b: of eight main variants, the long-only portfolio beats the unconstrained one net of cost in at least this many

# Notebook 11, the optimiser, fixed 4 October 2026 before its code; the
# constraints themselves date from 10 September. The problem, the infeasible-month
# rule and the expectations are stated in the notebook's introduction and in
# optimiser.py.
OPT_SOLVER = "CLARABEL"                  # interior-point solver shipped with cvxpy; deterministic
OPT_SOLVER_OPTIONS = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-10}   # tighter than the solver's defaults (1e-8), which left weights 1e-4 off an independent solver's on a test problem
OPT_SOLVER_OPTIONS_ROBUST = {"tol_gap_abs": 1e-9, "tol_gap_rel": 1e-9, "tol_feas": 1e-9}   # the robust problem is written with second-order cones, on which the solver reaches 1e-9 and reports 1e-12 as inaccurate; the objective agrees with the 1e-12 run to 1e-10 on the test problems
OPT_TOL = 1e-6                           # feasibility checks on weights, bounds and turnover; also "binding" means within this of the limit
OPT_TOL_TE = 1e-6                        # expectation 2a: a forecast tracking error below this, per year, counts as zero
OPT_TILT_TE_MAX_ANNUAL = 0.01            # expectation 3: 100 basis points a year, the enhanced-indexing ex-ante limit (Robeco's composite ran at 1.06%, J.P. Morgan quotes below 2%; sources in notebook 11)
OPT_REPORT_MONTHS = ("first_eval", "2004-11", "last")   # first_eval is COV_EVAL_START, the first month with both windows
OPT_SPEED_LIMIT_STEPS = (1, 6, 12, 24)   # expectation 5: rebalances reported on the journey from 1/N to the benchmark
# Checks D1 to D4 on the practical problem behind the tilt's hedge, added after
# notebook 11's first run (4 October 2026), each fixed here before its code.
OPT_D1_MARKET_SHARE_MAX = 0.05         # D1a: the market part of the tilt's forecast active variance is below this share in every month-estimator pair
OPT_D2_TE_RISE_MAX = 0.0001            # D2a: a beta-neutral constraint raises the forecast tracking error by less than one basis point a year in every pair
OPT_D3_EXCLUDE = ("Smoke",)            # D3: the hedge's tobacco purchase refused, its cost reported
# The fixes F1 to F3 (4 October 2026, before notebook 12's code): beta neutrality
# and the exclusions join C3 (above), and a robust portfolio, the weights whose
# largest forecast active variance across the estimators' covariances is
# smallest (Goldfarb and Iyengar 2003; Tutuncu and Koenig 2004), joins the
# rolling evaluation as a variant. Its beta neutrality holds under the average
# of the five beta vectors, because one equality per covariance was infeasible
# at 1979-07 under the mandate's bounds.
OPT_FIX_RISE_MAX = 0.0015              # F2: the mandate's C3 costs less than 15 basis points a year more than the tilt alone
ROBUST_VARIANT = ("robust", "all_five")   # the robust portfolio: minimax over the five covariances
# Check D5 on the solver, added after notebook 12's first run (6 October 2026),
# fixed here before its code: at this month the index moved more than the cap
# allows, no portfolio satisfied the cap and the tilt at once, and the solver
# reported "optimal" with weights that fail the independent check. The
# optimiser therefore finds the smallest feasible turnover first and never
# poses a capped problem that has no solution; the check of every returned
# portfolio stays.
OPT_D5_MONTH = "1984-07"

# Notebook 12, the rolling evaluation, fixed 4 October 2026 before its code: one
# path per specification, each portfolio built on the data before its month,
# judged on realised tracking error, calibration of the forecasts, active
# return net of costs, turnover and holdings. The expectations are in the
# notebook's introduction.
EVAL_TE_RANGE_MAX = 0.0010           # expectation 2: 10 basis points a year across the four daily estimators
EVAL_BIAS_RANGE = (0.8, 1.25)        # expectation 3, the same band as COV_SCALE_RATIO_RANGE
EVAL_RELAX_SHARE_MAX = 0.10          # expectation 6: at most one month in ten with a relaxed cap
EVAL_START_PORTFOLIO = "benchmark"   # every path starts as the index
EVAL_BENCHMARK_TOL = 1e-6            # expectation 1: an active return below this a month counts as holding the benchmark; written as 1e-10 and set to the solver's precision (about 1e-7) before the recorded run
# Expectations 8 and 9, added after notebook 12's first run (6 October 2026),
# each fixed here before its code and labelled in the notebook. RiskMetrics
# joins as a seventh variant because Dom, Howard, Jansen and Lohre (2024) find
# that time dynamics matter more than shrinkage or factor structure; the
# calibration multiplies each forecast by the ratio of realised to forecast
# tracking error over the previous EVAL_CALIBRATION_WINDOW months of the same
# path, using only months before the forecast.
EWMA_VARIANT = ("sample", "daily_5y_ewma")   # the seventh variant: RiskMetrics, as in notebook 10's sensitivity
EVAL_CALIBRATION_WINDOW = 60                 # months of a path's own past the calibration uses; a standard deviation from 60 observations has a standard error of about 9%
EVAL_CALIBRATION_MIN = 24                    # the forecast is left as it is while fewer months are available
EVAL_CALIBRATED_BIAS_RANGE = (0.85, 1.15)    # expectation 9: the calibrated bias statistic under C3, four daily estimators

# Notebook 13, the factor attribution, fixed 6 October 2026 before its code: each
# path's monthly active return split into the contributions of the six factors
# (the active exposure to each, from the industries' betas estimated on the data
# before the month, times the factor's return in the month) plus a residual, the
# industries' own returns; the identity is the check. The paths attributed are
# the seven variants under the mandate's C3 and under the tilt alone, rebuilt
# from the modules; the expectations are in the notebook's introduction.
ATTRIBUTION_FACTORS = FACTOR_MODEL_FACTORS
ATTRIBUTION_IDENTITY_MAX_ABS_ERROR = 1e-9        # expectation 1: contributions plus residual minus active return, per month, in return units
ATTRIBUTION_SETS = ("C3", "C3 tilt only")        # the two constraint sets whose paths are attributed
ATTRIBUTION_REPRODUCTION_TOL = 0.00001           # expectation 1: the rebuilt paths reproduce notebook 12's realised tracking errors within 0.1 basis point a year
ATTRIBUTION_REFERENCE_TE = {                     # notebook 12's realised tracking errors, per year, for the reproduction check
    ("sample, daily 3y", "C3"): 0.0084654, ("ledoit_wolf, daily 3y", "C3"): 0.0084820, ("factor, daily 3y", "C3"): 0.0093449,
    ("pca, daily 3y", "C3"): 0.0091482, ("sample, monthly 120", "C3"): 0.0095761, ("robust, all five", "C3"): 0.0087358,
    ("RiskMetrics, daily 5y", "C3"): 0.0084278,
    ("sample, daily 3y", "C3 tilt only"): 0.0080276, ("ledoit_wolf, daily 3y", "C3 tilt only"): 0.0080282, ("factor, daily 3y", "C3 tilt only"): 0.0088296,
    ("pca, daily 3y", "C3 tilt only"): 0.0086722, ("sample, monthly 120", "C3 tilt only"): 0.0090924, ("robust, all five", "C3 tilt only"): 0.0082305,
    ("RiskMetrics, daily 5y", "C3 tilt only"): 0.0079698,
}
ATTRIBUTION_MARKET_EXPOSURE_MAX = 0.05           # expectation 2: the mean active exposure to Mkt-RF, in absolute value, for every path
ATTRIBUTION_SIGN_AGREEMENT_MIN = 4               # expectation 3: of the five factors other than the market, the number whose mean active exposure under the mandate has the same sign for all seven variants
ATTRIBUTION_FACTOR_SHARE_MAX = 0.5               # expectation 4: the share of the active return's variance the six factors explain, for every path
ATTRIBUTION_FACTOR_PART_MAX = 0.5                # expectation 5a: the share of the mandate's cost against the tilt alone that comes through the factor contributions, every variant
ATTRIBUTION_ARITHMETIC_TOL = 0.0002              # expectation 5b: the exclusions' factor part (the five factors other than the market) within this of the arithmetic of the inputs, per year (2 basis points), every variant

# Specification count, written down before the code runs and reported.
EXT_SPEC_COUNT = (len(COV_ESTIMATORS) + 3) * len(CONSTRAINT_SETS)   # 28 optimised specifications: four daily-based estimators, the monthly sample covariance, the robust portfolio and RiskMetrics, times four constraint sets
EXT_COST_LEVELS = (DGU_TCOST, TCOST_SENSITIVITY)              # each reported net of cost at both levels; the cost is applied after the fact and does not enter the optimisation

# ---------------------------------------------------------------------------
# 6. Output conventions
# ---------------------------------------------------------------------------

OUTPUT_DIR = "outputs"
PROVENANCE_FILE = "outputs/provenance_french.json"   # download date, URL, SHA-256 and CRSP vintage line per file
TIMEZONE = "Europe/Amsterdam"

# ---------------------------------------------------------------------------
# 7. Version 2, module A: the factor objective under the mandate
# ---------------------------------------------------------------------------
# Fixed 8 October 2026, before any of the module's code was written, from the
# design of 7 October 2026 and version 1's results at commit 6835d49. Version 1 found that
# the mandate's tilt leaves incidental exposures of 0.004 to 0.018 in absolute
# value to the six factors, costing 9 to 13 basis points a year. Module A makes
# the exposures the objective: each month, with the data available before the
# month, the optimiser maximises the sum of the active exposures to the four
# targeted factors, s' B' (w - b), where B holds the industries' betas to the
# six factors estimated as in notebook 13 (three years of daily returns before
# the month) and s is FO_TARGET_SIGN, under every constraint of the mandate
# (CONSTRAINT_SETS["C3"]) and one new constraint, the budget: the forecast
# tracking error from the month's covariance estimate at most FO_BUDGETS_ANNUAL.
# The study holds no view on expected returns, as version 1 did not: the
# targets are the four of French's six factors with a documented premium, the
# direction of each is the literature's, and the four are weighted equally in
# beta units. Nothing estimates a premium or times a factor.
# Naming, fixed in the design: "tilt" stays the industry reduction of version 1;
# the new objective is the factor objective, the four are the targeted factors,
# the limit is the budget.
FO_DECISION_DATE = "2026-10-08"
FO_TARGET_FACTORS = ("HML", "RMW", "CMA", "Mom")              # value, profitability, investment, momentum
FO_TARGET_SIGN = {"Mkt-RF": 0.0, "SMB": 0.0, "HML": 1.0, "RMW": 1.0, "CMA": 1.0, "Mom": 1.0}   # s, on ATTRIBUTION_FACTORS; +1 is the direction of the documented premium
FO_CONSTRAINT_SET = "C3"                                       # the mandate: every constraint of version 1's C3, unchanged
FO_BUDGETS_ANNUAL = (0.0075, 0.0100, 0.0150)                   # tau, per year: just above the mandate's own realised cost (84.7 to 95.8 bp), the enhanced-indexing limit of OPT_TILT_TE_MAX_ANNUAL, and the generous case
FO_BUDGET_MAIN = 0.0100                                        # the budget at which the robust, RiskMetrics and single-factor paths run and most expectations are tested
FO_BETA_WINDOW_YEARS = EXT_COV_DAILY_WINDOW_YEARS              # the betas' window: notebook 13's three years of daily returns before the month
# Infeasibility rule: in a month in which the budget cannot be met under the
# mandate (the tracking-error-minimising solution of version 1 already exceeds
# tau), the path takes that minimising solution, the month is counted and
# reported, and nothing else is relaxed; the count is one of the results.
FO_INFEASIBLE_RULE = "tracking-error minimiser"
# The 19 paths, the count fixed here: the five estimator variants of version 1
# at each of the three budgets (15); the robust and RiskMetrics variants at the
# main budget (2); two single-factor sensitivities at the main budget with the
# daily sample estimator, s on value alone and on momentum alone (2). Version
# 1's mandate paths (tracking error minimised, no objective) are the comparison
# and are not rerun. A path added after the first run is disclosed with its
# date and reason, and the count restated.
FO_SINGLE_FACTOR_PATHS = (("HML",), ("Mom",))
FO_PATH_COUNT = 19
# The two measures new to module A. The exposure the budget bought: the mean
# over the months of the sum of the four targeted active exposures, divided by
# the realised tracking error in basis points a year. The gap: for each month,
# the targeted exposure the optimiser chose ex ante, s' B_t' a_t with B_t the
# betas estimated before the month, minus the exposure the same weights carry
# when the betas are estimated after the month, s' B_{t+1}' a_t; its mean
# absolute value over the months is the path's gap.
FO_GAP_DEFINITION = "s'(B_t - B_{t+1})' a_t, mean absolute value over the months"
# The seven expectations, each a strict pass or fail; the levels from
# version 1's results at 6835d49, with the design's reasons.
# E1 and E2 were restated on 9 October 2026, after notebook 14's check of
# the arithmetic and before any path ran, by the design of 9 October 2026:
# a level changed at that point, between the pre-run check notebook 14
# exists for and the first rolling run, is a pre-run restatement, dated and
# disclosed here and once in the write-up; a level changed after notebook 15
# runs is a post-run change. The original levels of 8 October stay below,
# marked as superseded.
FO_RESTATEMENT_DATE = "2026-10-09"
# E1. At the main budget the budget binds (forecast tracking error within
# FO_E1_BIND_TOL of the budget) in at least FO_E1_BIND_SHARE_MIN of the
# months in which the mandate's own minimum forecast tracking error, with
# turnover free, is at or below the budget, for each of the four daily
# estimators. The months the mandate alone cannot meet are reported per
# estimator and budget from notebook 14's feasibility table; the months the
# path misses beyond them, with the turnover cap in force, are counted and
# reported, since that difference is what the path adds. Reason for the
# restatement: notebook 14 found the mandate alone above 100 bp in 37
# (sample), 43 (Ledoit-Wolf) and 39 (six-factor) of the 566 months, 22 for
# principal components, nearly all in 1979 to 1989, so the budget cannot
# bind in those months whatever the objective.
FO_E1_BIND_TOL_ANNUAL = 0.0001           # one basis point a year
FO_E1_BIND_SHARE_MIN = 0.95
FO_E1_DENOMINATOR = "months the mandate alone can meet"
# Superseded on 9 October 2026: the same share of all 566 months. Reason
# given on 8 October: version 1's forecast under the mandate was 55 to 68 bp
# on average and 47 to 55 for the tilt alone at the 1979 oil peak for the
# daily estimators, so a budget of 100 should be reachable almost always;
# the monthly estimator (108 at that peak) may fail it in the first years.
FO_E1_DENOMINATOR_SUPERSEDED_2026_10_08 = "all 566 months"
# E2. At the main budget the mean active exposure to each targeted factor is
# positive on every main path, the average of the four is at least
# FO_E2_MEAN_EXPOSURE_MIN for the daily sample estimator, and that average
# rises with the budget (75, then 100, then 150) on every estimator; the four
# exposures are reported per path at every budget, the average never alone.
# Reason for the restatement: notebook 14's stationary mean of the four at
# 100 bp is 0.030 for the daily sample estimator and a capped path holds at
# most the stationary exposure on average; the level is four fifths of
# 0.030, the fifth being the design's allowance for what the turnover cap,
# the infeasible months and the rolling betas may cost against the
# stationary solution, a judgement stated as such. Under an active weight
# bound of 2 percentage points on 49 industries, 100 bp of budget buys about
# 0.03 of average exposure in beta units.
FO_E2_MEAN_EXPOSURE_MIN = 0.024
# Superseded on 9 October 2026. Reason given on 8 October: version 1's
# incidental exposures were 0.004 to 0.018 in absolute value with no
# objective; 49 industries within 2 percentage points of their weights can
# reach three to ten times that when the objective asks for it.
FO_E2_MEAN_EXPOSURE_MIN_SUPERSEDED_2026_10_08 = 0.05
# One diagnostic per path, counted and reported with no expectation
# attached (the design of 9 October 2026): the months in which any targeted
# exposure ends below its incidental value, the value the mandate's
# tracking-error minimiser carries. Notebook 14 found such a month in 5 of
# its 15 month-estimator pairs at 100 bp.
FO_DIAGNOSTIC_BELOW_INCIDENTAL = "months with a targeted exposure below its incidental value"
# E3. Realised tracking error exceeds the budget by 20 to 60 per cent on the
# daily estimators, and by more on the monthly sample covariance; tested at
# every budget (the design names none), the monthly ratio against the largest
# daily ratio at the same budget. Reason: version 1's bias was 1.22 to 1.39 on
# daily data and 1.61 on monthly; a budget-binding solution concentrates risk
# where the estimator sees least, so the miss may exceed version 1's, bounded
# here at 1.6.
FO_E3_REALISED_OVER_BUDGET_RANGE = (1.2, 1.6)
# E4. The four targeted factors' contributions sum to a positive amount on
# every main path, and at the main budget and above the six factors explain at
# least FO_E4_FACTOR_SHARE_MIN of the active return's variance on every main
# path. Reason: version 1's factors explained 14 to 28 per cent of incidental
# exposures; deliberate exposures raise the share.
FO_E4_FACTOR_SHARE_MIN = 0.30
# E5. At the main budget the gross active return is positive for at least
# FO_E5_POSITIVE_MIN of the five estimators; the net return at DGU_TCOST per
# unit of turnover lies within FO_E5_NET_GAP_MAX of the gross. Whether the
# active return is more than two standard errors from zero is reported, not
# expected. Reason: the study holds no view on the premia; the sign follows
# from the targets if the premia had their documented sign over the sample,
# which is what the data say and not a claim of skill.
FO_E5_POSITIVE_MIN = 4
FO_E5_NET_GAP_MAX = 0.0010               # 10 basis points a year
# E6. Of the two single-factor paths, momentum alone has the higher turnover
# and the larger gap, and the turnover cap binds in more months on the
# momentum path than on the value path. Reason: industry momentum betas rotate
# within a year; value betas do not.
# E7. At the main budget the spread across the four daily estimators in
# realised tracking error is at most FO_E7_TE_RANGE_MAX, the exposure bought
# per basis point differs across them by less than FO_E7_EXPOSURE_PER_BP_REL_MAX
# of their mean (largest minus smallest over the mean), and the monthly sample
# covariance remains the worst calibrated (the largest realised over forecast
# of the five). Reason: version 1, a range of 9 basis points on a level of
# about 90, the monthly covariance 61 per cent under-forecast.
FO_E7_TE_RANGE_MAX = 0.0020              # 20 basis points a year
FO_E7_EXPOSURE_PER_BP_REL_MAX = 0.30
# Notebook 14 runs the objective at the report months of version 1 and over
# the sample without a rolling path: a worked month, the exposure against the
# budget at FO_CURVE_BUDGETS (the three budgets and points between), and the
# feasibility of each budget in every month, from the forecast tracking error
# of the tracking-error-minimising solution under the mandate.
FO_CURVE_BUDGETS_ANNUAL = (0.0050, 0.0075, 0.0100, 0.0125, 0.0150, 0.0200)
FO_TOL_TE = 1e-9                         # a forecast tracking error within this of the budget counts as on it, in the tests

In [ ]:
%%writefile src/bp/french_loader.py
"""
Ken French data library: download, record, parse.

The library serves each dataset as a zip holding one CSV. The CSV is not a
single table. It is a text file with between two and ten blocks, each announced by a title
line ("  Average Value Weighted Returns -- Monthly"), followed by a header row
that starts with a comma, followed by rows that start with a date: six digits
for monthly data (192607), four for annual (1927). Missing values are coded
-99.99 or -999. Returns are in percent.

This module turns that into pandas DataFrames indexed by a monthly Period, in
decimal returns, with the missing codes replaced by NaN. It also writes a
provenance record for every file it downloads: the URL, the download time, the
SHA-256 of the zip, and the CRSP vintage line French prints at the top of each
CSV. A result is only as reproducible as the vintage it was computed on, and
French revises history, so the record is part of every output.

Nothing here is committed to the repository: the zips live in a local cache
directory that .gitignore excludes.
"""
from __future__ import annotations

import hashlib
import io
import json
import re
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests

from . import constants as C

_DATE_ROW = re.compile(r"^\s*(\d{8}|\d{6}|\d{4})\s*,")   # daily YYYYMMDD, monthly YYYYMM, annual YYYY


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def download(key: str, cache_dir: str | Path = "data/french", force: bool = False) -> dict:
    """Fetch one French zip by its key in constants.FRENCH_FILES.

    Returns the provenance record for the file and writes the zip to cache_dir.
    A cached zip is reused unless force=True, and its record is rebuilt from the
    cached bytes, so the SHA-256 always describes the file actually on disk.
    """
    filename = C.FRENCH_FILES[key]
    url = C.FRENCH_BASE_URL + filename
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / filename
    if force or not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        fetched_now = True
    else:
        fetched_now = False
    data = path.read_bytes()
    now_utc = datetime.now(timezone.utc)
    record = {
        "key": key,
        "filename": filename,
        "url": url,
        "fetched_this_run": fetched_now,
        "download_time_utc": now_utc.isoformat(timespec="seconds"),
        "download_time_local": now_utc.astimezone(ZoneInfo(C.TIMEZONE)).isoformat(timespec="seconds"),
        "zip_sha256": sha256_bytes(data),
        "zip_bytes": len(data),
        "vintage_line": _vintage_line(data),
    }
    return record


def _csv_text(zip_bytes: bytes) -> str:
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if len(names) != 1:
            raise ValueError(f"expected one CSV in the zip, found {names}")
        return zf.read(names[0]).decode("latin-1")


def _vintage_line(zip_bytes: bytes) -> str:
    """The first line of the CSV, e.g. 'This file was created using the 202607 CRSP database.'"""
    text = _csv_text(zip_bytes)
    for line in text.splitlines():
        if line.strip():
            first = line.strip()
            return first.split(".")[0] + "." if "." in first else first
    return ""


def parse_blocks(text: str) -> list[dict]:
    """Split a French CSV into its blocks.

    Each block is returned as {"title": str, "frequency": "daily" | "monthly" |
    "annual", "frame": DataFrame} where the frame is indexed by Period (day,
    month or year) and holds floats in the file's own units (percent). The title is the last non-empty line before
    the header row that is not itself a data row.
    """
    lines = text.splitlines()
    blocks = []
    i = 0
    n = len(lines)
    while i < n:
        if _DATE_ROW.match(lines[i]):
            # Walk back to the header row (starts with a comma) and the title above it.
            header_idx = i - 1
            while header_idx >= 0 and not lines[header_idx].lstrip().startswith(","):
                header_idx -= 1
            if header_idx < 0:
                raise ValueError(f"data row at line {i} has no header row above it")
            title_idx = header_idx - 1
            while title_idx >= 0 and not lines[title_idx].strip():
                title_idx -= 1
            title = lines[title_idx].strip() if title_idx >= 0 else ""
            # The factor files and the later blocks of the size-and-value file
            # carry no title line, only prose; a sentence is not a title.
            if not title or title.endswith(".") or len(title) > 70:
                title = f"(untitled block {len(blocks)})"
            columns = [c.strip() for c in lines[header_idx].split(",")[1:]]
            rows = []
            j = i
            while j < n and _DATE_ROW.match(lines[j]):
                rows.append(lines[j])
                j += 1
            raw = pd.read_csv(io.StringIO("\n".join(rows)), header=None, index_col=0)
            raw.columns = columns[: raw.shape[1]]
            date_len = len(str(raw.index[0]).strip())
            if date_len == 8:
                idx = pd.PeriodIndex(pd.to_datetime(raw.index.astype(str), format="%Y%m%d"), freq="D")
                freq = "daily"
            elif date_len == 6:
                idx = pd.PeriodIndex([pd.Period(f"{str(d)[:4]}-{str(d)[4:6]}", freq="M") for d in raw.index])
                freq = "monthly"
            else:
                idx = pd.PeriodIndex([pd.Period(str(d), freq="Y") for d in raw.index])
                freq = "annual"
            frame = raw.astype(float)
            frame.index = idx
            frame.index.name = "period"
            blocks.append({"title": title, "frequency": freq, "frame": frame})
            i = j
        else:
            i += 1
    return blocks


def load_blocks(key: str, cache_dir: str | Path = "data/french") -> list[dict]:
    """All blocks of one file, with the file's provenance record attached to each."""
    record = download(key, cache_dir)
    text = _csv_text((Path(cache_dir) / C.FRENCH_FILES[key]).read_bytes())
    blocks = parse_blocks(text)
    for b in blocks:
        b["provenance"] = record
    return blocks


def load_monthly(key: str, block: int | str = 0, cache_dir: str | Path = "data/french",
                 as_decimal: bool = True) -> pd.DataFrame:
    """One block of one file, in decimal returns with missing codes as NaN.

    The name dates from the monthly files; the function serves the daily files
    in the same way (the block's index is then a daily Period).

    block is an integer position among the file's blocks, or a regular
    expression matched against block titles (the first match wins).
    """
    blocks = load_blocks(key, cache_dir)
    if isinstance(block, int):
        chosen = blocks[block]
    else:
        matches = [b for b in blocks if re.search(block, b["title"], flags=re.I)]
        if not matches:
            raise KeyError(f"no block in {key} matches {block!r}; titles: {[b['title'] for b in blocks]}")
        chosen = matches[0]
    frame = chosen["frame"].copy()
    for code in C.FRENCH_MISSING_CODES:
        frame = frame.mask(np.isclose(frame, code))
    if as_decimal and C.FRENCH_RETURNS_ARE_PERCENT:
        frame = frame / 100.0
    frame.attrs["title"] = chosen["title"]
    frame.attrs["provenance"] = chosen["provenance"]
    return frame


def load_counts(key: str, block: int | str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """A block in the file's own units (firm counts, average firm size), missing codes as NaN."""
    return load_monthly(key, block, cache_dir, as_decimal=False)


def write_provenance(records: list[dict], path: str | Path = C.PROVENANCE_FILE) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ordered = sorted(records, key=lambda r: r["key"])
    path.write_text(json.dumps(ordered, indent=2, sort_keys=True) + "\n")
    return path


def block_summary(key: str, cache_dir: str | Path = "data/french") -> pd.DataFrame:
    """One row per block: title, frequency, columns, rows, first and last period."""
    rows = []
    for k, b in enumerate(load_blocks(key, cache_dir)):
        f = b["frame"]
        rows.append({
            "file": key, "block": k, "title": b["title"], "frequency": b["frequency"],
            "columns": f.shape[1], "rows": f.shape[0],
            "first": str(f.index[0]), "last": str(f.index[-1]),
        })
    return pd.DataFrame(rows)

In [ ]:
%%writefile src/bp/benchmark.py
"""The benchmark of the extension: the cap-weighted combination of French's 49 industry portfolios.

Each industry portfolio's value-weighted return for a month is the return of the
firms in the industry, each counted in proportion to its market equity at the
start of the month. Weighting the 49 industry returns by each industry's total
market equity at the same time (firm count times average firm size, two blocks of
the same French file) gives the value-weighted return of all the firms in the 49
portfolios, which should be close to French's market return. Notebook 08 measures
how close, against the level fixed in constants.CAPW_TE_MAX_ANNUAL, and decides
whether these weights are the benchmark of notebooks 09 to 13.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from . import constants as C
from . import french_loader as fl

PERIODS_PER_YEAR = 12


def load_inputs(cache_dir: str = "data/french") -> dict[str, pd.DataFrame | pd.Series]:
    """The three monthly blocks of the 49-industry file and the market series.

    returns: value-weighted industry returns (decimal); firms: number of firms;
    size: average firm size in millions of dollars; market: Mkt-RF plus RF from
    the factors file (decimal). Missing codes are NaN in all of them.
    """
    returns = fl.load_monthly("ind49", r"Value Weighted Returns -- Monthly", cache_dir)
    firms = fl.load_counts("ind49", r"Number of Firms", cache_dir)
    size = fl.load_counts("ind49", r"Average Firm Size", cache_dir)
    f3 = fl.load_monthly("factors3", 0, cache_dir)
    excess, rf = C.CAPW_MARKET_SERIES
    market = (f3[excess] + f3[rf]).rename("market")
    return {"returns": returns, "firms": firms, "size": size, "market": market}


def market_equity(firms: pd.DataFrame, size: pd.DataFrame) -> pd.DataFrame:
    """Total market equity of each industry, in the units of the size block (millions of dollars)."""
    if not firms.index.equals(size.index) or not firms.columns.equals(size.columns):
        raise ValueError("firm counts and average sizes must share months and industries")
    return firms * size


def cap_weights(firms: pd.DataFrame, size: pd.DataFrame, timing: str = C.CAPW_WEIGHT_TIMING) -> pd.DataFrame:
    """Each industry's share of total market equity, per month; the rows sum to one.

    timing "same_month": the count and size of month t weight the returns of
    month t (French measures both at the start of the month). "previous_month":
    the count and size of month t-1 weight the returns of month t; the first
    month has no weights. Any month in which an industry's count or size is
    missing has no weights at all, so that a partial sum never passes as a
    whole-market weight.
    """
    me = market_equity(firms, size)
    if timing == "previous_month":
        me = me.shift(1)
    elif timing != "same_month":
        raise ValueError(f"unknown timing {timing!r}")
    total = me.sum(axis=1, min_count=me.shape[1])
    return me.div(total, axis=0)


def combination_return(returns: pd.DataFrame, weights: pd.DataFrame) -> pd.Series:
    """Sum over industries of weight times return, per month; NaN where any of the 49 is missing."""
    if not returns.columns.equals(weights.columns):
        raise ValueError("returns and weights must have the same industries in the same order")
    aligned = weights.reindex(returns.index)
    return (aligned * returns).sum(axis=1, min_count=returns.shape[1]).rename("combination")


def tracking_error(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> float:
    """Annualised standard deviation of the per-period difference a - b, over the periods both have."""
    d = (a - b).dropna()
    if len(d) < 2:
        return float("nan")
    return float(d.std(ddof=1) * np.sqrt(periods_per_year))


def compare(a: pd.Series, b: pd.Series, periods_per_year: int = PERIODS_PER_YEAR) -> dict[str, float]:
    """Tracking error, annualised mean difference, correlation, largest absolute monthly difference and month count of a against b."""
    both = pd.concat([a, b], axis=1).dropna()
    d = both.iloc[:, 0] - both.iloc[:, 1]
    return {
        "months": int(len(d)),
        "tracking_error_annual": float(d.std(ddof=1) * np.sqrt(periods_per_year)),
        "mean_difference_annual": float(d.mean() * periods_per_year),
        "correlation": float(both.iloc[:, 0].corr(both.iloc[:, 1])) if both.std(ddof=1).gt(0).all() else float("nan"),
        "max_abs_monthly_difference": float(d.abs().max()),
        "month_of_max": str(d.abs().idxmax()),
    }


def weight_extremes(weights: pd.DataFrame, month: str | pd.Period) -> dict[str, object]:
    """The largest and smallest weights in one month, with their industries."""
    row = weights.loc[pd.Period(month, "M")].dropna()
    return {
        "month": str(pd.Period(month, "M")),
        "largest": row.idxmax(), "largest_weight": float(row.max()),
        "smallest": row.idxmin(), "smallest_weight": float(row.min()),
        "industries": int(len(row)),
    }

In [ ]:
%%writefile src/bp/pca.py
"""Principal components of a panel of excess returns (notebook 09).

A covariance matrix of N assets holds N(N+1)/2 numbers. Its principal components
are the combinations of assets that account for most of the shared movement,
ordered by how much each explains: the first eigenvector of the covariance matrix
is the direction of largest variance, the second the direction of largest variance
among those uncorrelated with the first, and so on. Each eigenvalue is the variance
of the corresponding component, and the eigenvalues sum to the total variance of
the panel. The functions here return the components with their signs fixed by a
rule (an eigenvector and its negative are the same direction), the share of total
variance each explains, the components' monthly returns (their scores), a stability
measure between two sets of loadings, a parallel analysis against pure noise, and
a rolling view of the first component.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from . import constants as C


@dataclass
class PCA:
    """Eigenvalues (descending), loadings (one column per component, rows in the order of the assets), variance shares and the asset names."""
    eigenvalues: np.ndarray
    loadings: np.ndarray
    shares: np.ndarray
    assets: list[str]
    months: int

    def frame(self, n: int | None = None) -> pd.DataFrame:
        n = n or len(self.eigenvalues)
        return pd.DataFrame(self.loadings[:, :n], index=self.assets, columns=[f"PC{k + 1}" for k in range(n)])


def covariance(excess: pd.DataFrame) -> np.ndarray:
    """The sample covariance of the columns (denominator T-1), on the months with no missing value."""
    X = excess.dropna().to_numpy(dtype=float)
    return np.cov(X, rowvar=False, ddof=1)


def fit(excess: pd.DataFrame, reference: pd.Series | np.ndarray | None = None) -> PCA:
    """Principal components of the covariance of `excess` (months by assets).

    reference: a weight vector over the assets (the benchmark's weights, for
    example) used by the sign rule: each component is flipped, if needed, so
    that its loading on the reference portfolio (loadings dot reference) is
    positive. With no reference the rule uses 1/N weights, so that a
    component whose loadings are mostly positive keeps them positive.
    """
    clean = excess.dropna()
    Sigma = np.cov(clean.to_numpy(dtype=float), rowvar=False, ddof=1)
    vals, vecs = np.linalg.eigh(Sigma)               # ascending
    order = np.argsort(vals)[::-1]
    vals, vecs = vals[order], vecs[:, order]
    vals = np.clip(vals, 0.0, None)                   # rounding can give -1e-20
    ref = np.ones(Sigma.shape[0]) / Sigma.shape[0] if reference is None else np.asarray(reference, dtype=float)
    for k in range(vecs.shape[1]):
        if vecs[:, k] @ ref < 0:
            vecs[:, k] = -vecs[:, k]
    return PCA(eigenvalues=vals, loadings=vecs, shares=vals / vals.sum(), assets=list(excess.columns), months=len(clean))


def scores(excess: pd.DataFrame, pca: PCA, n: int | None = None) -> pd.DataFrame:
    """The components' scores: their monthly returns, the demeaned excess returns projected on the loadings."""
    n = n or len(pca.eigenvalues)
    clean = excess.dropna()
    X = clean.to_numpy(dtype=float)
    X = X - X.mean(axis=0)
    S = X @ pca.loadings[:, :n]
    return pd.DataFrame(S, index=clean.index, columns=[f"PC{k + 1}" for k in range(n)])


def loading_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """The absolute correlation between two loading vectors: 1 means the same combination of assets, 0 means unrelated."""
    return float(abs(np.corrcoef(np.asarray(a, dtype=float), np.asarray(b, dtype=float))[0, 1]))


def explained_in_window(excess_window: pd.DataFrame, loadings: np.ndarray, n: int) -> float:
    """The share of a window's total variance that the first n given loading vectors explain inside that window.

    The share of the window's own components is the largest possible for n
    components; a lower share for loadings estimated elsewhere measures how much
    the structure moved.
    """
    Sigma = covariance(excess_window)
    L = loadings[:, :n]
    Q, _ = np.linalg.qr(L)                            # orthonormal basis of the same subspace
    return float(np.trace(Q.T @ Sigma @ Q) / np.trace(Sigma))


def parallel_analysis(excess: pd.DataFrame, draws: int = C.PCA_PARALLEL_DRAWS, percentile: float = C.PCA_PARALLEL_PERCENTILE,
                      seed: int = C.PCA_PARALLEL_SEED) -> pd.DataFrame:
    """How many components exceed what pure noise would produce.

    Draws `draws` panels of independent normal returns with the same number of
    months and assets and the same variances as the data (so no shared movement
    at all), takes the eigenvalues of each, and reports for each component the
    data's eigenvalue, the noise percentile and whether the data exceed it. The
    count of components above the bar is the number of components that carry
    structure the noise cannot imitate.
    """
    clean = excess.dropna()
    X = clean.to_numpy(dtype=float)
    T, N = X.shape
    sd = X.std(axis=0, ddof=1)
    rng = np.random.default_rng(seed)
    noise_vals = np.empty((draws, N))
    for d in range(draws):
        Z = rng.standard_normal((T, N)) * sd
        noise_vals[d] = np.sort(np.linalg.eigvalsh(np.cov(Z, rowvar=False, ddof=1)))[::-1]
    bar = np.percentile(noise_vals, percentile, axis=0)
    data_vals = np.sort(np.linalg.eigvalsh(np.cov(X, rowvar=False, ddof=1)))[::-1]
    out = pd.DataFrame({"eigenvalue": data_vals, f"noise_p{int(percentile)}": bar, "above_noise": data_vals > bar},
                       index=[f"PC{k + 1}" for k in range(N)])
    out.attrs["n_above_noise"] = int(out["above_noise"].to_numpy().cumprod().sum())   # the leading run of components above the bar
    return out


def rolling_first_component(excess: pd.DataFrame, benchmark_excess: pd.Series, window: int = C.EXT_ESTIMATION_WINDOW,
                            reference: np.ndarray | None = None) -> pd.DataFrame:
    """For every window of `window` months ending at each month: the first component's variance share and its correlation with the benchmark's excess return inside the window."""
    rows = []
    idx = excess.index
    bench = benchmark_excess.reindex(idx)
    for end in range(window - 1, len(idx)):
        sl = excess.iloc[end - window + 1: end + 1]
        if sl.isna().any().any():
            continue
        p = fit(sl, reference)
        s1 = scores(sl, p, 1)["PC1"]
        rows.append({"month": idx[end], "pc1_share": float(p.shares[0]), "pc1_corr_benchmark": float(np.corrcoef(s1, bench.iloc[end - window + 1: end + 1])[0, 1]),
                     "first_five_share": float(p.shares[:5].sum())})
    return pd.DataFrame(rows).set_index("month")

## Count first: the panel of excess returns and the benchmark

The industries' value-weighted returns minus the one-month Treasury bill rate, from 1969-07, and the benchmark of notebook 08 rebuilt from the same module.

In [ ]:
import sys
sys.path.insert(0, "src")
import numpy as np
import pandas as pd
from bp import constants as C
from bp import french_loader as fl
from bp import benchmark as BM
from bp import pca as P

pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

inputs = BM.load_inputs()
f3 = fl.load_monthly("factors3", 0)
start = pd.Period(C.EXT_SAMPLE_START, "M")
returns = inputs["returns"].loc[start:]
rf = f3["RF"].loc[start:returns.index[-1]]
excess = returns.sub(rf, axis=0)
weights = BM.cap_weights(inputs["firms"].loc[start:], inputs["size"].loc[start:], C.CAPW_WEIGHT_TIMING)
benchmark = BM.combination_return(returns, weights)
benchmark_excess = (benchmark - rf).rename("benchmark_excess")
LAST = excess.index[-1]

assert excess.shape[1] == 49 and not excess.isna().any().any(), "a gap in the excess returns"
assert len(excess) == len(pd.period_range(start, LAST, freq="M")), "months missing"
assert benchmark_excess.index.equals(excess.index) and benchmark_excess.notna().all(), "the benchmark does not cover the same months"
assert rf.notna().all()
records = [fl.download(key) for key in ("ind49", "factors3")]
fl.write_provenance(records)
reference = weights.mean().to_numpy()      # the sign rule: each component's loading on the average benchmark weights is positive
print(f"{len(excess)} months, {start} to {LAST}, 49 industries, no gap; benchmark rebuilt over the same months (expectation 1 holds)")
print(f"covariance matrix: 49 variances and {49 * 48 // 2:,} covariances, {49 * 50 // 2:,} numbers; a 120-month window holds {49 * 120:,} monthly returns, {49 * 120 / (49 * 50 // 2):.1f} per number")
print("vintage:", "; ".join(sorted({r["vintage_line"] for r in records})))

## The components over the full sample, and whether the first is the market

The first table lists the first ten components: the variance each explains, its share of the total, and the cumulative share. The variance and standard deviation columns are in the component's own units: the loadings are scaled so that their squares sum to one, which makes the first component's monthly return about seven times the average industry's excess return (its 49 loadings are each about 0.14 and add up to about 7), so only ratios between components and correlations carry meaning, and the variance share is the ratio that matters. The last line of the cell is expectation 2 over the full sample: the correlation between the first component's score and the benchmark's excess return.

In [ ]:
full = P.fit(excess, reference)
table = pd.DataFrame({
    "variance (monthly)": full.eigenvalues[:C.PCA_REPORT_COMPONENTS],
    "standard deviation (monthly)": np.sqrt(full.eigenvalues[:C.PCA_REPORT_COMPONENTS]),
    "variance share": full.shares[:C.PCA_REPORT_COMPONENTS],
    "cumulative share": np.cumsum(full.shares)[:C.PCA_REPORT_COMPONENTS],
}, index=[f"PC{k + 1}" for k in range(C.PCA_REPORT_COMPONENTS)])
print(table.to_string())
print()
print(f"total variance of the 49 industries: {full.eigenvalues.sum():.5f} a month; the first {C.PCA_N_COMPONENTS} components explain {full.shares[:C.PCA_N_COMPONENTS].sum():.1%} of it, the first alone {full.shares[0]:.1%}")
print(f"the average industry has a monthly standard deviation of {np.sqrt(np.diag(P.covariance(excess))).mean():.4f}; the first component's is {np.sqrt(full.eigenvalues[0]):.4f}")
sc = P.scores(excess, full, C.PCA_N_COMPONENTS)
corr_full = float(np.corrcoef(sc["PC1"], benchmark_excess)[0, 1])
print()
print(f"expectation 2, full sample: correlation between the first component's score and the benchmark's excess return {corr_full:.4f} "
      f"(level {C.PCA_PC1_MARKET_CORR_MIN}): {'MET' if corr_full >= C.PCA_PC1_MARKET_CORR_MIN else 'NOT MET'}")
print("correlation of each of the first five scores with the benchmark's excess return:", {c: round(float(np.corrcoef(sc[c], benchmark_excess)[0, 1]), 3) for c in sc.columns})

## What the first component is made of

Each industry's loading on the first component, beside two things that could explain it: the industry's own monthly standard deviation and its average benchmark weight. If the first component were the benchmark, the loadings would follow the weights. If it is the market factor as a statistical object, they follow how much each industry moves with the market, which is larger for volatile industries whatever their size.

In [ ]:
L = full.frame(C.PCA_N_COMPONENTS)
sd = pd.Series(np.sqrt(np.diag(P.covariance(excess))), index=excess.columns, name="monthly standard deviation")
avg_w = weights.mean().rename("average benchmark weight")
beta = pd.Series({c: np.cov(excess[c], benchmark_excess, ddof=1)[0, 1] / benchmark_excess.var(ddof=1) for c in excess.columns}, name="beta to the benchmark")
view = pd.concat([L["PC1"].rename("PC1 loading"), sd, beta, avg_w], axis=1).sort_values("PC1 loading", ascending=False)
print("the ten largest and the ten smallest loadings on the first component:")
print(pd.concat([view.head(10), view.tail(10)]).to_string())
print()
print(f"correlation across industries between the PC1 loading and: the standard deviation {view['PC1 loading'].corr(view['monthly standard deviation']):.2f}, "
      f"the beta to the benchmark {view['PC1 loading'].corr(view['beta to the benchmark']):.2f}, the benchmark weight {view['PC1 loading'].corr(view['average benchmark weight']):.2f}")
print(f"loadings on PC1 that are negative: {int((L['PC1'] < 0).sum())} of 49")
print()
print("the second and third components, five largest positive and five most negative loadings each:")
for c in ("PC2", "PC3"):
    s_ = L[c].sort_values()
    print(f"  {c}: positive {{{', '.join(f'{k}: {v:.3f}' for k, v in s_.tail(5)[::-1].items())}}}; negative {{{', '.join(f'{k}: {v:.3f}' for k, v in s_.head(5).items())}}}")

## Expectation 2 in the three windows fixed in advance

The same fit inside each window, and the first component's correlation with the benchmark's excess return there.

In [ ]:
def window_slice(frame, lo, hi):
    return frame.loc[(pd.Period(lo, "M") if lo else frame.index[0]):(pd.Period(hi, "M") if hi else frame.index[-1])]

fits, rows = {}, []
for name, (lo, hi) in C.PCA_WINDOWS.items():
    ex = window_slice(excess, lo, hi)
    be = window_slice(benchmark_excess, lo, hi)
    p = P.fit(ex, reference)
    s1 = P.scores(ex, p, 1)["PC1"]
    corr = float(np.corrcoef(s1, be)[0, 1])
    fits[name] = (ex, p)
    rows.append({"window": name, "months": len(ex), "first month": str(ex.index[0]), "last month": str(ex.index[-1]),
                 "PC1 variance share": p.shares[0], f"first {C.PCA_N_COMPONENTS} share": p.shares[:C.PCA_N_COMPONENTS].sum(),
                 "PC1 corr. with benchmark": corr, "verdict": "MET" if corr >= C.PCA_PC1_MARKET_CORR_MIN else "NOT MET"})
e2 = pd.DataFrame(rows).set_index("window")
print(e2.to_string())
n_met = int((e2["verdict"] == "MET").sum())
print()
print(f"expectation 2: {n_met} of {len(e2)} windows at or above {C.PCA_PC1_MARKET_CORR_MIN}, and the full sample {'MET' if corr_full >= C.PCA_PC1_MARKET_CORR_MIN else 'NOT MET'}")
if n_met < len(e2) or corr_full < C.PCA_PC1_MARKET_CORR_MIN:
    print("design consequence: notebook 10's PCA estimator imposes the market factor as its first component")
else:
    print("design consequence: none; notebook 10's PCA estimator takes its components from the data")

## Expectation 3: how many components are more than noise

Parallel analysis draws 200 panels of independent normal returns with the same number of months and industries and the same variances as the data, so that nothing moves together in them, and takes the variance of each component in each draw. The bar for a component is the 95th percentile of its variance across the draws. A component of the data above its bar carries shared movement that independent noise does not produce one time in twenty.

In [ ]:
pa = P.parallel_analysis(excess)
show = pa.head(C.PCA_REPORT_COMPONENTS).copy()
show["ratio to the bar"] = show["eigenvalue"] / show[f"noise_p{C.PCA_PARALLEL_PERCENTILE}"]
print(show.to_string())
print()
print(f"components above the noise bar, counted from the first until the first one below it: {pa.attrs['n_above_noise']} of 49 "
      f"(the estimator of notebook 10 keeps {C.PCA_N_COMPONENTS})")
print(f"components above the bar anywhere in the list: {int(pa['above_noise'].sum())}")

## Expectation 4: do the components of one period still describe another

Two views. The first table gives, for each of the first five components, the similarity of its loadings between each pair of windows: the absolute correlation across the 49 industries of the two loading vectors, 1 for the same combination of industries and 0 for unrelated ones. The second table asks the question the estimator cares about: inside each window, how much of the variance do the full-sample components explain, against how much the window's own components explain, for one component and for five. The window's own components are the best possible for that window, so the gap between the two columns is the cost of using components estimated on other months.

In [ ]:
names = list(C.PCA_WINDOWS)
pairs = [(names[0], names[1]), (names[1], names[2]), (names[0], names[2])]
sim = pd.DataFrame({f"{a} vs {b}": [P.loading_similarity(fits[a][1].loadings[:, k], fits[b][1].loadings[:, k]) for k in range(C.PCA_N_COMPONENTS)] for a, b in pairs},
                   index=[f"PC{k + 1}" for k in range(C.PCA_N_COMPONENTS)])
print("similarity of loadings between windows (absolute correlation across the 49 industries):")
print(sim.to_string())
print()
rows = []
for name, (ex, p) in fits.items():
    rows.append({"window": name,
                 "own PC1": P.explained_in_window(ex, p.loadings, 1), "full-sample PC1": P.explained_in_window(ex, full.loadings, 1),
                 f"own first {C.PCA_N_COMPONENTS}": P.explained_in_window(ex, p.loadings, C.PCA_N_COMPONENTS),
                 f"full-sample first {C.PCA_N_COMPONENTS}": P.explained_in_window(ex, full.loadings, C.PCA_N_COMPONENTS)})
ex4 = pd.DataFrame(rows).set_index("window")
ex4["gap, five components"] = ex4[f"own first {C.PCA_N_COMPONENTS}"] - ex4[f"full-sample first {C.PCA_N_COMPONENTS}"]
print("share of each window's variance explained by its own components and by the full-sample components:")
print(ex4.to_string())

## Expectation 5: what the estimator will see, window by window

The first component's variance share and its correlation with the benchmark's excess return in every 120-month window, 567 of them, summarised by their minimum, median and maximum with the months at which the extremes end.

In [ ]:
roll = P.rolling_first_component(excess, benchmark_excess, C.EXT_ESTIMATION_WINDOW, reference)
summary = pd.DataFrame({
    "minimum": roll.min(), "month of minimum": roll.idxmin().astype(str), "median": roll.median(),
    "maximum": roll.max(), "month of maximum": roll.idxmax().astype(str),
})
print(f"{len(roll)} windows of {C.EXT_ESTIMATION_WINDOW} months, ending {roll.index[0]} to {roll.index[-1]}:")
print(summary.to_string())
print()
below = roll[roll["pc1_corr_benchmark"] < C.PCA_PC1_MARKET_CORR_MIN]
print(f"windows in which the first component's correlation with the benchmark is below {C.PCA_PC1_MARKET_CORR_MIN}: {len(below)} of {len(roll)}"
      + (f", ending {below.index[0]} to {below.index[-1]}" if len(below) else ""))
print("first component's variance share by decade of the window's last month (median):")
print((roll["pc1_share"].groupby((roll.index.year // 10) * 10).median()).to_string())

## What this notebook established, and what could be wrong

| Expectation | Result | Verdict |
|---|---|---|
| 1. Count first | 686 months from 1969-07 to 2026-08, 49 industries, no gap; the benchmark rebuilt over the same months | holds |
| 2. The first component is the market: correlation at least 0.95 with the benchmark's excess return, full sample and three windows | 0.956 over the full sample; 0.959 to 2019-12, 0.971 in 2020 to 2022, 0.837 from 2023-01 | NOT MET in the window from 2023; the pre-registered consequence applies |
| 3. Components above the parallel-analysis bar | 2 | reported |
| 4. Stability of the loadings between windows | first component similarity 0.53 to 0.68; the full-sample components explain 0.2, 9.3 and 11.4 points less than each window's own | reported |
| 5. The rolling view, 567 windows of 120 months | first component share 37% to 69%, median 61%; correlation with the benchmark below 0.95 in 130 windows, all ending after 2000-05 | reported |

Expectation 1 holds: 686 months from 1969-07 to 2026-08, 49 industries, no gap, and the benchmark rebuilt over the same months.

What the components are. The first component explains 55% of the total variance of the 49 industries over the full sample, the second 6%, the third 4%, and the first five together 72%. Every one of the 49 loadings on the first component is positive: when it moves, all industries move the same way, which is what a market factor does. Its loadings follow how much each industry moves with the market and not how large the industry is: across industries the loading has a correlation of 0.97 with the industry's beta to the benchmark and of -0.30 with its benchmark weight. Software, Steel, Fun, Construction and Real Estate load most; Utilities, Food, Tobacco, Drugs and Gold load least. The second component sets the commodity producers (Gold, Coal, Mines, Oil, Steel, all with negative loadings) against retail, software, entertainment and clothing, so it is the component that moves when raw-material prices move against the rest of the economy; the third separates Gold from Coal.

Expectation 2, the first component is the market: met over the full sample (correlation 0.956) and in the windows to 2019-12 (0.959) and 2020-01 to 2022-12 (0.971), and not met in the window from 2023-01 (0.837, 44 months). The loadings table explains the miss. The benchmark weights industries by size, and since 2023 Chips and Software hold a third of it between them; the first component weights industries by how much they move with the others, which gives Chips and Software about 3% each of a component whose loadings are all between 0.07 and 0.20. In a period in which the largest industries move on their own, the two series part. The rolling view says the same: in 130 of the 567 windows of 120 months, all of them ending after May 2000, the correlation is below 0.95, with a minimum of 0.93 in the window ending December 2000. The pre-registered consequence applies, and `constants.py` records it: notebook 10's PCA estimator takes the benchmark's excess return as its first factor and finds the remaining components in what the market leaves unexplained, so that its first component is the market by construction in every window.

Expectation 3, how many components are more than noise: two. The first component's variance is 8.7 times what 200 panels of independent returns with the same variances produce at the 95th percentile; the second is 1.15 times; the third is below the bar, the fourth 1.02 times and the fifth below. The number the estimator of notebook 10 keeps, five, was fixed on 10 September; this count says that components three to five carry about as much shared movement as chance would, over the full sample. The constant stays at five, and notebook 10 reports the two-component estimator beside it, with neither declared right or wrong.

Expectation 4, stability. The first component's loadings have a similarity of 0.68 between the window to 2019 and the window 2020 to 2022, 0.61 between the second and the third, and 0.53 between the first and the third; the loadings of components two to five are mostly unrelated between windows (0.02 to 0.55). The second table asks what that costs. Inside the window to 2019 the full-sample components explain 72.9% of the variance against 73.1% for the window's own, a gap of 0.2 points, because that window is most of the full sample. Inside 2020 to 2022 the gap is 9.3 points (73.7% against 83.0%) and inside the window from 2023 it is 11.4 points (61.9% against 73.3%). The first component alone moves less: gaps of 0.1, 2.1 and 4.8 points. So the combinations of industries that moved together in one period explain a good part of another, and the smaller components are the ones that do not carry over.

Expectation 5, the rolling view. Across the 567 windows of 120 months, the first component's variance share runs from 37% (the window ending February 2001, the ten years of the technology boom, when technology moved on its own) to 69% (the window ending December 1987, which holds the crash of October 1987, a month in which everything fell together), with a median of 61%; by decade of the window's last month the median is 67% in the 1970s, 41% in the 2000s and 55% in the 2020s. The first five components explain between 66% and 84%.

**What this notebook does not settle.**

- The 44-month window from 2023 is short: a correlation measured on 44 months has a standard error of about 0.05 when its true value is near 0.9, so 0.837 against the level of 0.95 is two standard errors below it, and the rolling view confirms that the fall is real and not a feature of one short window.
- A variance share says how much of the industries' movement a component explains over the months it was estimated on; how well a covariance built from it serves a portfolio in the following month is notebook 10's question, and the gaps of expectation 4 are the first measure of it.
- Parallel analysis compares the data with independent returns of the same variances and nothing else; returns with fat tails or volatility clustering (notebooks 05 and 06) produce slightly larger extreme eigenvalues by chance, so the bar is slightly too low and the count of two is a generous one.
- The count of five components is a design parameter with a convention behind it (statistical risk models keep three to ten), fixed before the run, and the data-driven rules for the count disagree with each other: the Kaiser rule (components of the correlation matrix with variance above one), the scree plot (where the ordered variances flatten), parallel analysis (where chance would produce as much, two here), the Marchenko-Pastur bound of random-matrix theory (the same idea as a formula) and the information criteria of Bai and Ng (2002). All of them judge how well the components describe the sample. The count that serves a portfolio is the one whose covariance matrix forecasts next month's risk best, found by running the estimator with each count month by month on the months before each forecast; notebook 10 runs two counts beside each other and reports the difference, and a rule that chooses the count inside each window is the data-driven version a later study could fix before its run.
- The components were extracted from the covariance of excess returns, so an industry with twice the volatility weighs four times as much in the total variance; extracting them from the correlation matrix instead would treat every industry alike and give a different second and third component. The choice was fixed before the run because the estimator of notebook 10 works with covariances.